--------------------------------------------------------------
Section 1: Data Update in Excel Tables
--------------------------------------------------------------

In [25]:
# ===============================================================================
# STEP 1: READ THE EXCEL FILE (already downloaded by EU_Fuel_Price_ETL Notebook1)
# ===============================================================================

print("File ready ✅")

import pandas as pd

File ready ✅


In [26]:
# ============================================================
# STEP 2: COUNTRY DICTIONARY
# Key = country code in the file / Value = full name in English
# ============================================================

country_names = {
    'AT_': 'Austria',
    'BE_': 'Belgium',
    'BG_': 'Bulgaria',
    'CY_': 'Cyprus',
    'CZ_': 'Czech Republic',
    'DE_': 'Germany',
    'DK_': 'Denmark',
    'EE_': 'Estonia',
    'ES_': 'Spain',
    'FI_': 'Finland',
    'FR_': 'France',
    'GR_': 'Greece',
    'HR_': 'Croatia',
    'HU_': 'Hungary',
    'IE_': 'Ireland',
    'IT_': 'Italy',
    'LT_': 'Lithuania',
    'LU_': 'Luxembourg',
    'LV_': 'Latvia',
    'MT_': 'Malta',
    'NL_': 'Netherlands',
    'PL_': 'Poland',
    'PT_': 'Portugal',
    'RO_': 'Romania',
    'SE_': 'Sweden',
    'SI_': 'Slovenia',
    'SK_': 'Slovakia'
}

In [27]:
# ============================================================
# STEP 3: EXCHANGE RATES BY COUNTRY (local currency to euros)
# Countries using the euro have a rate of 1
# ============================================================

exchange_rates = {
    'Austria'        : 1,      # EUR
    'Belgium'        : 1,      # EUR
    'Bulgaria'       : 1.9558, # BGN
    'Cyprus'         : 1,      # EUR
    'Czech Republic' : 24.282, # CZK
    'Denmark'        : 7.4733, # DKK
    'Estonia'        : 1,      # EUR
    'Finland'        : 1,      # EUR
    'France'         : 1,      # EUR
    'Germany'        : 1,      # EUR
    'Greece'         : 1,      # EUR
    'Croatia'        : 1,      # EUR
    'Hungary'        : 353.84, # HUF
    'Ireland'        : 1,      # EUR
    'Italy'          : 1,      # EUR
    'Latvia'         : 1,      # EUR
    'Lithuania'      : 1,      # EUR
    'Luxembourg'     : 1,      # EUR
    'Malta'          : 1,      # EUR
    'Netherlands'    : 1,      # EUR
    'Poland'         : 4.3,    # PLN
    'Portugal'       : 1,      # EUR
    'Romania'        : 4.95,   # RON
    'Slovakia'       : 1,      # EUR
    'Slovenia'       : 1,      # EUR
    'Spain'          : 1,      # EUR
    'Sweden'         : 11.5    # SEK
}

In [28]:
# ============================================================
# STEP 4: READ THE 3 SHEETS OF THE EXCEL FILE
# ============================================================

df_vat        = pd.read_excel("/lakehouse/default/Files/oil_bulletin_duties_and_taxes.xlsx", sheet_name=0, header=None)
df_excise     = pd.read_excel("/lakehouse/default/Files/oil_bulletin_duties_and_taxes.xlsx", sheet_name=1, header=None)
df_othertaxes = pd.read_excel("/lakehouse/default/Files/oil_bulletin_duties_and_taxes.xlsx", sheet_name=2, header=None)

In [29]:
# ============================================================
# STEP 5: CLEAN EACH DATAFRAME
# ============================================================

def clean(df):
    df = df.iloc[4:].copy()
    df = df.iloc[:, [0, 2, 3]]
    df.columns = ['Country', 'SP95', 'Diesel']

    # Replace country codes with full names
    df['Country'] = df['Country'].map(country_names)

    # Remove rows where country is empty (disclaimer at the bottom)
    df = df.dropna(subset=['Country'])

    return df

df_vat        = clean(df_vat)
df_excise     = clean(df_excise)
df_othertaxes = clean(df_othertaxes)


In [30]:
# ============================================================
# STEP 6: CONVERT TO EUROS (only sheets 2 and 3)
# ============================================================

def convert_to_euros(df):
    df['SP95']   = df['SP95']   / df['Country'].map(exchange_rates)
    df['Diesel'] = df['Diesel'] / df['Country'].map(exchange_rates)
    return df

df_excise     = convert_to_euros(df_excise)
df_othertaxes = convert_to_euros(df_othertaxes)

In [31]:
# ============================================================
# STEP 7: RENAME COLUMNS AND RESET INDEX
# ============================================================

df_vat = df_vat.rename(columns={
    'Country' : 'Country',
    'SP95'    : 'VAT_Petrol',
    'Diesel'  : 'VAT_Diesel'
}).reset_index(drop=True)

df_excise = df_excise.rename(columns={
    'Country' : 'Country',
    'SP95'    : 'Excise_Petrol',
    'Diesel'  : 'Excise_Diesel'
}).reset_index(drop=True)

df_othertaxes = df_othertaxes.rename(columns={
    'Country' : 'Country',
    'SP95'    : 'Other_Taxes_Petrol',
    'Diesel'  : 'Other_Taxes_Diesel'
}).reset_index(drop=True)

In [32]:
# ============================================================
# STEP 8: CONVERT PRICES FROM 1000L TO 1L
# ============================================================

df_excise['Excise_Petrol']          = df_excise['Excise_Petrol']          / 1000
df_excise['Excise_Diesel']          = df_excise['Excise_Diesel']          / 1000

df_othertaxes['Other_Taxes_Petrol'] = df_othertaxes['Other_Taxes_Petrol'] / 1000
df_othertaxes['Other_Taxes_Diesel'] = df_othertaxes['Other_Taxes_Diesel'] / 1000

In [33]:
# ============================================================
# STEP 9: CONVERT VAT TO PERCENTAGE
# ============================================================

df_vat['VAT_Petrol'] = df_vat['VAT_Petrol'] / 100
df_vat['VAT_Diesel'] = df_vat['VAT_Diesel'] / 100

In [34]:
# ============================================================
# STEP 10: DISPLAY RESULTS
# ============================================================

print("=== VAT ===")
print(df_vat)

print("\n=== EXCISE ===")
print(df_excise)

print("\n=== OTHER TAXES ===")
print(df_othertaxes)

=== VAT ===
           Country VAT_Petrol VAT_Diesel
0          Austria        0.2        0.2
1          Belgium       0.21       0.21
2         Bulgaria        0.2        0.2
3           Cyprus       0.19       0.19
4   Czech Republic       0.21       0.21
5          Germany       0.19       0.19
6          Denmark       0.25       0.25
7          Estonia       0.22       0.22
8            Spain        0.1        0.1
9          Finland      0.255      0.255
10          France        0.2        0.2
11          Greece       0.24       0.24
12         Croatia       0.25       0.25
13         Hungary       0.27       0.27
14         Ireland       0.23       0.23
15           Italy       0.22       0.22
16       Lithuania       0.21       0.21
17      Luxembourg       0.17       0.17
18          Latvia       0.21       0.21
19           Malta       0.18       0.18
20     Netherlands       0.21       0.21
21          Poland       0.08       0.08
22        Portugal       0.23       0.23
23  

In [35]:
# ============================================================
# STEP 11: EXPORT TO EXCEL
# ============================================================

df_vat.to_excel('/lakehouse/default/Files/df_vat.xlsx', index=False)
df_excise.to_excel('/lakehouse/default/Files/df_excise.xlsx', index=False)
df_othertaxes.to_excel('/lakehouse/default/Files/df_othertaxes.xlsx', index=False)

print("Export complete!")

Export complete!


-------------------------------------------------------------------
SECTION 2: Data Update in Lakehouse Tables
-------------------------------------------------------------------

In [36]:
import os
import json
import uuid
import shutil
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
import pandas as pd

# ============================================================
# LOAD FILES FROME ONELAKE
# ============================================================
df_vat = pd.read_excel("/lakehouse/default/Files/df_vat.xlsx")
df_excise = pd.read_excel("/lakehouse/default/Files/df_excise.xlsx")
df_othertaxes = pd.read_excel("/lakehouse/default/Files/df_othertaxes.xlsx")

print("✅ Files uploaded")
print(df_vat.head())
print(df_excise.head())
print(df_othertaxes.head())

# ============================================================
# DELTA TABLE CREATION FUNCTION
# ============================================================
def create_delta_table(df, table_path, schema_arrow):
    os.makedirs(table_path, exist_ok=True)
    os.makedirs(f"{table_path}/_delta_log", exist_ok=True)

    arrow_table = pa.Table.from_pandas(df, schema=schema_arrow, preserve_index=False)
    parquet_filename = "data.parquet"
    parquet_path = f"{table_path}/{parquet_filename}"
    pq.write_table(arrow_table, parquet_path)

    def pa_type_to_delta(t):
        if pa.types.is_timestamp(t): return "timestamp"
        if pa.types.is_int64(t) or pa.types.is_int32(t): return "long"
        if pa.types.is_float64(t) or pa.types.is_float32(t): return "double"
        if pa.types.is_date(t): return "date"
        return "string"

    schema = arrow_table.schema
    delta_schema = {
        "type": "struct",
        "fields": [
            {"name": f.name, "type": pa_type_to_delta(f.type), "nullable": True, "metadata": {}}
            for f in schema
        ]
    }

    commit = [
        json.dumps({"protocol": {"minReaderVersion": 1, "minWriterVersion": 2}}),
        json.dumps({"metaData": {
            "id": str(uuid.uuid4()),
            "format": {"provider": "parquet", "options": {}},
            "schemaString": json.dumps(delta_schema),
            "partitionColumns": [],
            "createdTime": int(datetime.now().timestamp() * 1000),
            "configuration": {}
        }}),
        json.dumps({"add": {
            "path": parquet_filename,
            "size": os.path.getsize(parquet_path),
            "partitionValues": {},
            "modificationTime": int(datetime.now().timestamp() * 1000),
            "dataChange": True,
            "stats": json.dumps({"numRecords": len(df)})
        }})
    ]

    with open(f"{table_path}/_delta_log/00000000000000000000.json", "w") as f:
        f.write("\n".join(commit))

    print(f"✅ Table updated: {table_path}")

# ============================================================
# dim_VAT_bis
# ============================================================
df_vat_export = df_vat[['Country', 'VAT_Petrol', 'VAT_Diesel']].copy()
df_vat_export['Country'] = df_vat_export['Country'].astype(str)
df_vat_export['VAT_Petrol'] = df_vat_export['VAT_Petrol'].astype(float)
df_vat_export['VAT_Diesel'] = df_vat_export['VAT_Diesel'].astype(float)

schema_vat = pa.schema([
    pa.field("Country", pa.string()),
    pa.field("VAT_Petrol", pa.float64()),
    pa.field("VAT_Diesel", pa.float64()),
])

create_delta_table(df_vat_export, "/lakehouse/default/Tables/dim_VAT_bis", schema_vat)

# ============================================================
# dim_Excise_bis
# ============================================================
df_excise_export = df_excise[['Country', 'Excise_Petrol', 'Excise_Diesel']].copy()
df_excise_export['Country'] = df_excise_export['Country'].astype(str)
df_excise_export['Excise_Petrol'] = df_excise_export['Excise_Petrol'].astype(float)
df_excise_export['Excise_Diesel'] = df_excise_export['Excise_Diesel'].astype(float)

schema_excise = pa.schema([
    pa.field("Country", pa.string()),
    pa.field("Excise_Petrol", pa.float64()),
    pa.field("Excise_Diesel", pa.float64()),
])

create_delta_table(df_excise_export, "/lakehouse/default/Tables/dim_Excise_bis", schema_excise)

# ============================================================
# dim_other_taxes_bis
# ============================================================
df_othertaxes_export = df_othertaxes[['Country', 'Other_Taxes_Petrol', 'Other_Taxes_Diesel']].copy()
df_othertaxes_export['Country'] = df_othertaxes_export['Country'].astype(str)
df_othertaxes_export['Other_Taxes_Petrol'] = df_othertaxes_export['Other_Taxes_Petrol'].astype(float)
df_othertaxes_export['Other_Taxes_Diesel'] = df_othertaxes_export['Other_Taxes_Diesel'].astype(float)

schema_othertaxes = pa.schema([
    pa.field("Country", pa.string()),
    pa.field("Other_Taxes_Petrol", pa.float64()),
    pa.field("Other_Taxes_Diesel", pa.float64()),
])

create_delta_table(df_othertaxes_export, "/lakehouse/default/Tables/dim_other_taxes_bis", schema_othertaxes)

print("\n🎉 Complete export!")


✅ Files uploaded
          Country  VAT_Petrol  VAT_Diesel
0         Austria        0.20        0.20
1         Belgium        0.21        0.21
2        Bulgaria        0.20        0.20
3          Cyprus        0.19        0.19
4  Czech Republic        0.21        0.21
          Country  Excise_Petrol  Excise_Diesel
0         Austria       0.462000       0.377000
1         Belgium       0.600160       0.600160
2        Bulgaria       0.185612       0.168877
3          Cyprus       0.429000       0.400000
4  Czech Republic       0.528787       0.329915
          Country  Other_Taxes_Petrol  Other_Taxes_Diesel
0         Austria             0.13192             0.14546
1         Belgium             0.00000             0.00000
2        Bulgaria             0.00000             0.00000
3          Cyprus             0.01070             0.01070
4  Czech Republic             0.00000             0.00000
✅ Table updated: /lakehouse/default/Tables/dim_VAT_bis
✅ Table updated: /lakehouse/default/Tabl